### Phase 2: Data Preparation

In [1]:
import pandas as pd 
import numpy as np 
import warnings

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('../final_datasets/combined_datasets/era5_glaciers_datasets.csv')
df.shape

(13036, 70)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13036 entries, 0 to 13035
Data columns (total 70 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            13036 non-null  int64  
 1   glacier_id            13036 non-null  float64
 2   year                  13036 non-null  int64  
 3   glacier_name          13036 non-null  str    
 4   country_left          13036 non-null  str    
 5   region                13036 non-null  str    
 6   area_change           12847 non-null  float64
 7   elevation_change      12891 non-null  float64
 8   elevation_change_unc  12797 non-null  float64
 9   volume_change         12112 non-null  float64
 10  volume_change_unc     12106 non-null  float64
 11  index_right           13036 non-null  int64  
 12  rgi_id                13036 non-null  str    
 13  cenlon                13036 non-null  float64
 14  cenlat                13036 non-null  float64
 15  utm_zone              13036 no

In [4]:
df.describe()

,Unnamed: 0,glacier_id,year,area_change,elevation_change,elevation_change_unc,volume_change,volume_change_unc,index_right,cenlon,...,ssrd,strd,ro,es,sro,e,u10,v10,sp,tp
count,13036.000000,13036.000000,13036.000000,1.284700e+04,12891.000000,12797.000000,1.211200e+04,1.210600e+04,13036.000000,13036.000000,...,1.303600e+04,1.303600e+04,13036.000000,13036.000000,13036.000000,13036.000000,13036.000000,13036.000000,13036.000000,13036.000000
mean,6522.625499,65871.162933,2011.779994,1.214815e+06,-3.450149,4.546824,-3.480722e+06,3.469595e+06,3258.134474,84.295800,...,2.004194e+07,2.058994e+07,0.002860,-0.000076,0.001678,-0.000938,0.157905,0.063500,57100.868629,0.003950
std,3764.461738,22097.160113,6.028961,3.411525e+06,4.012567,1.876932,1.342450e+07,7.306259e+06,1836.748540,2.131604,...,1.479595e+06,1.773101e+06,0.001808,0.000037,0.001317,0.000371,0.288818,0.339151,3610.492856,0.001718
min,3.000000,904.000000,1952.000000,1.800000e+04,-175.400000,0.100000,-4.253925e+08,1.250000e+05,428.000000,80.810252,...,1.443644e+07,9.975764e+06,0.000193,-0.000206,0.000048,-0.002070,-0.401817,-0.441600,49889.045417,0.000924
25%,3262.750000,73731.000000,2009.000000,1.550000e+05,-4.805000,3.510000,-2.595000e+06,8.250000e+05,1770.750000,82.519025,...,1.899482e+07,1.949108e+07,0.001550,-0.000101,0.000755,-0.001183,-0.045667,-0.179411,54410.009750,0.002658
50%,6522.500000,74495.000000,2014.000000,3.730000e+05,-2.730000,4.180000,-8.700000e+05,1.530000e+06,2881.000000,83.931406,...,2.017458e+07,2.049751e+07,0.002465,-0.000074,0.001314,-0.000950,0.109870,-0.056364,57092.794500,0.003593
75%,9782.250000,75295.000000,2019.000000,9.590000e+05,-1.215000,5.292500,-2.650000e+05,3.189375e+06,4989.250000,86.499996,...,2.118343e+07,2.180212e+07,0.003805,-0.000048,0.002266,-0.000751,0.321154,0.230586,59284.454917,0.004904
max,13042.000000,216960.000000,2025.000000,6.105400e+07,30.017500,78.180000,6.113500e+07,1.348200e+08,6906.000000,88.176722,...,2.332857e+07,2.608420e+07,0.014379,0.000062,0.009601,-0.000019,1.606236,1.334671,70079.878333,0.014543


In [5]:
print(f"Before removing dupliate\n Shape : {df.shape}")
print(f"After removing dupliate\n Shape : {df.drop_duplicates().shape}")

Before removing dupliate
 Shape : (13036, 70)
After removing dupliate
 Shape : (13036, 70)


No duplicate rows present here

In [6]:
# viewing missing percentages of each feature
df.isna().mean().mul(100).sort_values(ascending=False)

glac_name               97.852102
volume_change_unc        7.134090
volume_change            7.088064
elevation_change_unc     1.833384
area_change              1.449831
                          ...    
e                        0.000000
u10                      0.000000
v10                      0.000000
sp                       0.000000
tp                       0.000000
Length: 70, dtype: float64

Around ~98% of data are missing in `glac_name` and ~7.2% in `volumne_change_unc` and `volume_change`.

Now, removing `glac_name` because of larger number of missing values, but interploating other minor missing values

In [7]:
missing_percent = df.isna().mean() * 100

# missing values over 90%
missing_90_percent_cols = missing_percent[missing_percent > 90].index.to_list()
print(f"Features/Columns having more than 90% missing values: {missing_90_percent_cols}")


# missing values greater than 0% but less than 90%
missing_zero_percent_cols = missing_percent[(missing_percent > 0) & (missing_percent < 90)].index.to_list()
print(f"Features/Columns having > 0% and < 90% missing values: {missing_zero_percent_cols}")

Features/Columns having more than 90% missing values: ['glac_name']
Features/Columns having > 0% and < 90% missing values: ['area_change', 'elevation_change', 'elevation_change_unc', 'volume_change', 'volume_change_unc']


In [8]:
df.drop(columns=missing_90_percent_cols, inplace=True)
df[missing_zero_percent_cols] = df[missing_zero_percent_cols].interpolate(method='linear', limit_direction='both')

# rechecking missing percentages|
df.isna().mean().mul(100).sort_values(ascending=False)


Unnamed: 0      0.0
glacier_id      0.0
year            0.0
glacier_name    0.0
country_left    0.0
               ... 
e               0.0
u10             0.0
v10             0.0
sp              0.0
tp              0.0
Length: 69, dtype: float64

None of the features have `NAN` (null) values

Now, removing some unnecessary features that have no roles such as "IDs" (feature selection).

In [15]:
df['primeclass'].value_counts()

primeclass
0    13036
Name: count, dtype: int64

`primeclass` has only one value i.e 0, which has no variance.

In [18]:
df['glacier_name'].value_counts()

glacier_name
RIKHA SAMBA         27
YALA                25
MERA                18
THULAGI             15
POKALDE             14
                    ..
AMADABLAM            1
LUMDING              1
UNNAMED_HIMALAYA     1
UNNAMED 18524        1
THYANBO              1
Name: count, Length: 3052, dtype: int64

In [10]:
print(df.columns.tolist())

['Unnamed: 0', 'glacier_id', 'year', 'glacier_name', 'country_left', 'region', 'area_change', 'elevation_change', 'elevation_change_unc', 'volume_change', 'volume_change_unc', 'index_right', 'rgi_id', 'cenlon', 'cenlat', 'utm_zone', 'area_km2', 'primeclass', 'conn_lvl', 'surge_type', 'term_type', 'zmin_m', 'zmax_m', 'zmed_m', 'zmean_m', 'slope_deg', 'aspect_deg', 'aspect_sec', 'dem_source', 'lmax_m', 'country_right', 'distance_degrees', 'd2m', 't2m', 'skt', 'stl1', 'stl2', 'stl3', 'stl4', 'lblt', 'licd', 'lict', 'lmld', 'lmlt', 'lshf', 'ltlt', 'asn', 'snowc', 'rsn', 'sde', 'sd', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'str', 'sshf', 'ssrd', 'strd', 'ro', 'es', 'sro', 'e', 'u10', 'v10', 'sp', 'tp']


In [ ]:
unnecessary_cols = ["Unnamed: 0", "glacier_id", "rgi_id", "index_right", "primeclass", "country_left", "country_right", "utm_zone", "conn_lvl"  ]

Now, handling outliers

In [16]:
# getting numerical features
numerical_cols = df.select_dtypes(include=['int', 'float']).columns.tolist()
print(numerical_cols)

['Unnamed: 0', 'glacier_id', 'year', 'area_change', 'elevation_change', 'elevation_change_unc', 'volume_change', 'volume_change_unc', 'index_right', 'cenlon', 'cenlat', 'utm_zone', 'area_km2', 'primeclass', 'conn_lvl', 'surge_type', 'term_type', 'zmin_m', 'zmax_m', 'zmed_m', 'zmean_m', 'slope_deg', 'aspect_deg', 'aspect_sec', 'lmax_m', 'distance_degrees', 'd2m', 't2m', 'skt', 'stl1', 'stl2', 'stl3', 'stl4', 'lblt', 'licd', 'lict', 'lmld', 'lmlt', 'lshf', 'ltlt', 'asn', 'snowc', 'rsn', 'sde', 'sd', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'str', 'sshf', 'ssrd', 'strd', 'ro', 'es', 'sro', 'e', 'u10', 'v10', 'sp', 'tp']


In [19]:
Q1 = df[numerical_cols].quantile(0.25)
Q3 = df[numerical_cols].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - (1.5 * IQR)
upper_limit = Q3 + (1.5 * IQR)

In [20]:
print("Dataset shape")
print(f"Before removing outliers: {df.shape}")

df_outliers_removal = df[(df[numerical_cols] >= lower_limit) & (df[numerical_cols] <= upper_limit)]

print(f"After removing outliers: {df_outliers_removal.shape}")

Dataset shape
Before removing outliers: (13036, 69)
After removing outliers: (13036, 69)


Dataset shape remains the constant before and after removing outliers, so  this means the dataset has no outliers.

In [22]:
categorical_cols = df.select_dtypes(include=['category', 'str']).columns.tolist()
print(categorical_cols)

['glacier_name', 'country_left', 'region', 'rgi_id', 'dem_source', 'country_right']
